# Kreo Digital · Session 08
# Q-Learning for Fraud Review

**Question:** With only ten review slots for the next 200 transactions, should we review this transaction or preserve capacity for a later one?

Session 7 ranked unusual transactions. Here we learn a **sequential review policy**. The agent sends transactions for review or passes them. It does not block payments or declare people fraudulent.

**Learning outcomes**
1. Describe a state, action, reward, episode and policy in a review workflow.
2. Calculate and implement one Q-learning update.
3. Apply epsilon-greedy exploration and enforce review capacity.
4. Compare a frozen policy with simple baselines on a chronological holdout.
5. Explain why simulated reward, fraud recall and real operational value differ.

**Prerequisites:** basic Python, NumPy indexing, train/validation/test splits. Session 7 is useful background; its outputs are not required. Suggested teaching time: 2–3 hours, including exercises.

> This is an educational simulation on PaySim, which itself contains simulated mobile-money transactions. Historical labels supply simulated feedback. Rewards are invented utility points, not measured money or evidence that reviewing prevents loss.

## 1 · From prediction to decisions

| Concept | In this lesson |
|---|---|
| Agent | The policy selecting transactions for analyst review |
| Environment | A replay of recorded transactions with a simulated budget and rewards |
| State | Current type, amount bin, remaining review slots, position in episode |
| Action | `PASS=0` or `REVIEW=1` |
| Reward | Utility points assigned after each simulated decision |
| Episode | 200 consecutive sampled transactions, starting with 10 slots |
| Q(s,a) | Estimated discounted return after taking action a in state s and following a greedy policy later |
| Policy | A rule choosing an available action from the observed state |

Both actions advance to the next transaction. REVIEW also consumes one slot. That shared capacity makes a decision affect future available actions. When capacity is zero, only PASS is legal. An episode ends after its last transaction, even if capacity runs out earlier.

These are consecutive rows **within a uniform sample**, not an uninterrupted production stream. Capacity resets every window by teaching convention, not every day or bank shift. Current features are coarse: future behavior can depend on history omitted from the state. This is an approximate state representation, not a proven Markov model of fraud.

**Discuss:** Why would independent review/pass decisions without a shared budget be closer to a contextual bandit than this resource-constrained problem?

## 2 · Rewards and feedback assumptions

| Action | Revealed label | Utility |
|---|---|---:|
| REVIEW | Fraud | +10 |
| REVIEW | Legitimate | −1 |
| PASS | Fraud | −10 |
| PASS | Legitimate | 0 |

A reviewed fraud receives a benefit; a legitimate review incurs analyst cost; passing fraud incurs a missed-fraud penalty. These are **illustrative** values. Changing them changes the task. Review does not itself prove a fraud or guarantee prevention.

For training, the simulator reveals the recorded label immediately after **every** action, including PASS. That allows rewards to be calculated; it does not make labels valid inputs to the state. In a real system, labels are delayed and often only investigated transactions receive reliable outcomes. This lesson assumes away that selective-feedback problem and does not provide an off-policy evaluation or deployment solution.

For a single isolated transaction with known fraud probability p, REVIEW has expected reward `11p − 1`, while PASS has `−10p`. The immediate reward advantage is `21p − 1`. Capacity and future value can make the sequential choice different. Neither the amount bin nor Q-values are calibrated fraud probabilities.

In [ ]:
from pathlib import Path
import os, json, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
TRAIN_SEEDS = [42, 43, 44]
MAX_ROWS = 50_000
EPISODE_LENGTH, REVIEW_BUDGET = 200, 10
EPOCHS, GAMMA = 30, 0.99
ALPHAS = [0.1, 0.3]
# Supports launching from either repository root or session-08.
SESSION = Path.cwd() if Path.cwd().name == 'session-08' else Path.cwd() / 'session-08'
DATA_PATH = Path(os.environ.get('KREO_DATA_PATH', str(
    SESSION.parent / 'session-07/data/PS_20174392719_1491204439457_log.csv'))).expanduser()
if not DATA_PATH.is_file():
    local = SESSION / 'data/PS_20174392719_1491204439457_log.csv'
    if local.is_file(): DATA_PATH = local
    else: raise FileNotFoundError(f'PaySim not found: {DATA_PATH}. See session-08/README.md or set KREO_DATA_PATH.')
OUTPUT = SESSION / 'outputs'; OUTPUT.mkdir(exist_ok=True)
versions = {p: importlib.metadata.version(p) for p in ['numpy','pandas','matplotlib','nbformat','nbclient']}
plt.rcParams.update({'figure.figsize': (10,4), 'axes.grid': True})
print('Python:', platform.python_version(), versions)
print('CSV:', DATA_PATH.resolve())

## 3 · An inspectable learning core

Read `ReviewEnv.step` and `q_update` first; the other functions organize experiments. They live here so you can inspect the complete algorithm without an RL framework.

The Q-table is a dictionary mapping the tuple `(type, amount_bin, remaining_slots, position)` to two numbers `[Q_pass, Q_review]`. Unseen states start at zero. Greedy ties choose PASS; training exploration may try either legal action. Evaluation never creates or updates table entries. This conservative fallback can miss fraud in unseen states, so we report state coverage.

Amount bins are quartiles of **training** log amount. Type is a fixed known PaySim category list. No fraud label, account name, balance field or existing fraud flag is included in policy inputs.

In [ ]:
import numpy as np
import pandas as pd

PASS, REVIEW = 0, 1
TYPE_ORDER = ['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']
REWARDS = {(REVIEW, 1): 10.0, (REVIEW, 0): -1.0,
           (PASS, 1): -10.0, (PASS, 0): 0.0}

class ReviewEnv:
    """Recorded context replay; labels are private to the reward simulator."""
    def __init__(self, contexts, labels, budget):
        self.contexts = np.asarray(contexts)
        self.labels = np.asarray(labels)
        if (self.contexts.ndim != 2 or self.contexts.shape[1] != 2
            or len(self.contexts) == 0 or self.labels.shape != (len(self.contexts),)
            or not np.isin(self.labels, [0, 1]).all()
            or not np.isfinite(self.contexts).all()
            or (self.contexts < 0).any() or (self.contexts % 1 != 0).any()
            or not isinstance(budget, (int, np.integer)) or not 0 <= budget <= len(self.contexts)):
            raise ValueError('Need nonempty integer contexts, aligned binary labels and a feasible integer budget.')
        self.contexts = self.contexts.astype(int)
        self.initial_budget = int(budget)
        self.reset()

    def reset(self):
        self.position, self.remaining = 0, self.initial_budget
        return self.state()

    def state(self):
        if self.position == len(self.contexts): return None
        kind, amount_bin = self.contexts[self.position]
        return (int(kind), int(amount_bin), self.remaining, self.position)

    def step(self, action):
        if self.state() is None: raise RuntimeError('Episode ended; call reset().')
        if action not in (PASS, REVIEW) or (action == REVIEW and self.remaining == 0):
            raise ValueError('Action unavailable in this state.')
        reward = REWARDS[(int(action), int(self.labels[self.position]))]
        self.remaining -= int(action == REVIEW)
        self.position += 1
        done = self.position == len(self.contexts)
        return self.state(), reward, done


def choose_action(q, state, rng, epsilon=0.0):
    if not 0 <= epsilon <= 1: raise ValueError('epsilon must be between 0 and 1.')
    if state[2] == 0: return PASS
    if epsilon > 0 and rng.random() < epsilon:
        return int(rng.integers(2))
    # np.argmax picks PASS on equal values; get() does not mutate unseen states.
    return int(np.argmax(q.get(state, np.zeros(2))))


def q_update(q, state, action, reward, next_state, done, alpha=0.1, gamma=0.99):
    if action not in (PASS, REVIEW) or (action == REVIEW and state[2] == 0):
        raise ValueError('Cannot learn an unavailable action.')
    if not 0 < alpha <= 1 or not 0 <= gamma <= 1:
        raise ValueError('Invalid learning rate or discount.')
    values = q.setdefault(state, np.zeros(2, dtype=float))
    bootstrap = 0.0
    if not done:
        if next_state is None: raise ValueError('Nonterminal transition needs next_state.')
        nxt = q.get(next_state, np.zeros(2))
        bootstrap = float(nxt[PASS] if next_state[2] == 0 else np.max(nxt))
    target = reward + gamma * bootstrap
    values[action] += alpha * (target - values[action])
    return float(values[action])


def prepare_data(frame):
    ordered = frame.sort_values('step', kind='stable').copy()
    if len(ordered) < 20: raise ValueError('Need more transactions for chronological splits.')
    cut1, cut2 = np.quantile(ordered.step, [.6, .8]).astype(int)
    parts = {'train': ordered.loc[ordered.step <= cut1].copy(),
             'validation': ordered.loc[(ordered.step > cut1) & (ordered.step <= cut2)].copy(),
             'test': ordered.loc[ordered.step > cut2].copy()}
    if any(part.empty for part in parts.values()):
        raise ValueError('Insufficient distinct step groups for three splits.')
    edges = np.unique(np.quantile(np.log1p(parts['train'].amount), [.25, .5, .75]))
    contexts = {}
    for name, part in parts.items():
        kinds = pd.Categorical(part.type, categories=TYPE_ORDER).codes
        if (kinds < 0).any(): raise ValueError('Unexpected transaction type.')
        bins = np.searchsorted(edges, np.log1p(part.amount), side='right')
        contexts[name] = np.column_stack([kinds, bins]).astype(int)
    assert parts['train'].step.max() < parts['validation'].step.min()
    assert parts['validation'].step.max() < parts['test'].step.min()
    return parts, contexts, edges, (int(cut1), int(cut2))


def episode_count(n, episode_length, budget):
    if (not isinstance(episode_length, (int, np.integer)) or episode_length < 1
        or not isinstance(budget, (int, np.integer)) or not 0 <= budget <= episode_length):
        raise ValueError('Need positive integer episode length and feasible integer budget.')
    count = n // episode_length
    if count == 0: raise ValueError('Split too short for a complete episode; increase sample size.')
    return count


def train_q(contexts, labels, episode_length=200, budget=10, epochs=30,
            alpha=0.1, gamma=0.99, seed=42):
    n_episodes = episode_count(len(contexts), episode_length, budget)
    if epochs < 1: raise ValueError('Need at least one training epoch.')
    rng = np.random.default_rng(seed)
    q, history = {}, []
    for epoch in range(epochs):
        epsilon = max(0.05, 1.0 * (0.90 ** epoch))
        total_reward, reviews = 0.0, 0
        # Shuffle episode order only; each episode remains chronological.
        for ep in rng.permutation(n_episodes):
            sl = slice(ep*episode_length, (ep+1)*episode_length)
            env = ReviewEnv(contexts[sl], labels[sl], budget)
            state = env.reset()
            while state is not None:
                action = choose_action(q, state, rng, epsilon)
                nxt, reward, done = env.step(action)
                q_update(q, state, action, reward, nxt, done, alpha, gamma)
                total_reward += reward; reviews += action
                state = nxt
        history.append({'epoch': epoch+1, 'epsilon': epsilon,
                        'mean_episode_reward': total_reward/n_episodes,
                        'mean_episode_reviews': reviews/n_episodes,
                        'visited_states': len(q)})
    return q, pd.DataFrame(history)


def evaluate_policy(contexts, labels, amounts, policy, q=None, threshold=None,
                    episode_length=200, budget=10, seed=42):
    if policy not in {'q','never','random','first','threshold','offline_amount'}:
        raise ValueError('Unknown policy.')
    if len(labels) != len(contexts) or len(amounts) != len(contexts):
        raise ValueError('Evaluation arrays must align.')
    if policy == 'threshold' and threshold is None:
        raise ValueError('Threshold policy needs a training-fitted threshold.')
    n_episodes = episode_count(len(contexts), episode_length, budget)
    rng = np.random.default_rng(seed)
    q = {} if q is None else q
    rows = []
    for ep in range(n_episodes):
        start, end = ep*episode_length, (ep+1)*episode_length
        env = ReviewEnv(contexts[start:end], labels[start:end], budget)
        # Random fixed positions need no labels or future features. Equivalent
        # online rule: review with probability remaining_slots/remaining_rows.
        selected = set(rng.choice(episode_length, size=budget, replace=False).tolist()) if policy == 'random' else set()
        if policy == 'offline_amount':
            selected = set(np.argsort(-np.asarray(amounts[start:end]), kind='stable')[:budget].tolist())
        state = env.reset()
        while state is not None:
            pos, before = env.position, env.remaining
            if before == 0: action = PASS
            elif policy == 'q': action = choose_action(q, state, rng, 0.0)
            elif policy == 'never': action = PASS
            elif policy == 'first': action = REVIEW
            elif policy == 'threshold': action = int(amounts[start+pos] >= threshold)
            else: action = int(pos in selected)
            nxt, reward, done = env.step(action)
            rows.append({'episode': ep, 'position': pos, 'row_position': start+pos,
                         'type_code': state[0], 'amount_bin': state[1],
                         'budget_before': before, 'budget_after': env.remaining,
                         'action': action, 'reward': reward,
                         'fraud_label_evaluation_only': int(labels[start+pos]),
                         'seen_state': state in q if policy == 'q' else np.nan})
            state = nxt
    trace = pd.DataFrame(rows)
    reviewed = trace.action.eq(REVIEW)
    y = trace.fraud_label_evaluation_only
    found, total_fraud = int(y[reviewed].sum()), int(y.sum())
    reviews = int(reviewed.sum())
    violations = int((trace.groupby('episode').action.sum() > budget).sum())
    metrics = {'evaluated_rows': len(trace), 'episodes': n_episodes,
               'dropped_rows': len(contexts)-len(trace), 'utility': float(trace.reward.sum()),
               'reviews': reviews, 'precision': found/reviews if reviews else np.nan,
               'recall': found/total_fraud if total_fraud else np.nan,
               'frauds_found': found, 'missed_fraud': total_fraud-found,
               'budget_violations': violations,
               'unseen_state_share': float((~trace.seen_state.astype(bool)).mean()) if policy == 'q' else np.nan}
    assert violations == 0 and (trace.budget_after >= 0).all()
    return metrics, trace

## 4 · The update, by hand

\[
Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma\max_{a'\in A(s')}Q(s',a') - Q(s,a)\right]
\]

- **α (alpha):** how much to move toward the current target.
- **γ (gamma):** how strongly to value later rewards. At 0.99, rewards many steps away have reduced weight. Evaluation below reports undiscounted utility separately.
- **ε (epsilon):** probability of a random legal training action, rather than the current greedy action. It is zero in evaluation.
- **Off-policy:** the update targets the greedy maximum even while the behavior policy explores.
- **Terminal transition:** there is no future term. Do not bootstrap beyond the episode.

Example: old `Q(s,PASS)=2`, reward `r=3`, next legal maximum `4`, `alpha=0.5`, `gamma=0.9`. Target = `3+0.9×4=6.6`; updated value = `2+0.5×(6.6−2)=4.3`.

If the next budget is zero, a stored `Q_review=999` must be ignored: that action is unavailable. Constants, repeated replay and approximate states here do **not** establish the assumptions needed for an asymptotic convergence theorem.

In [ ]:
s, nxt = (0, 0, 1, 0), (0, 0, 0, 1)
example_q = {s: np.array([2.0, 0.0]), nxt: np.array([4.0, 999.0])}
updated = q_update(example_q, s, PASS, 3, nxt, False, alpha=.5, gamma=.9)
assert np.isclose(updated, 4.3)
print('Worked update:', updated)

# A resource example, not a trained fraud result: one slot and two transactions.
toy = ReviewEnv([[0,0],[4,3]], [0,1], budget=1)
toy_rows=[]
for action in [PASS, REVIEW]:
    before = toy.state(); after, reward, done = toy.step(action)
    toy_rows.append({'state_before': str(before), 'action': ['PASS','REVIEW'][action],
                     'reward': reward, 'slots_after': toy.remaining, 'terminal': done})
display(pd.DataFrame(toy_rows))
print('Pass then review utility: +10; review then pass utility: -1 + -10 = -11.')
print('Labels shown for explaining the simulator; the policy does not observe them before acting.')

## 5 · Load and validate actual PaySim

We scan the entire CSV in chunks and retain the 50,000 rows with the smallest random priorities. Priorities are generated without labels. Sampling is uniform; fraud is not oversampled. The scan validates all rows and computes a source hash.

Sort retained rows chronologically (stable source order within equal steps). Use approximate 60/20/20 boundaries, keeping every row at a step on one side. Training rewards may use training labels; validation utility selects alpha. Final-test labels are accessed only for the final frozen comparison.

Account IDs, balance fields and the existing rule flag are validated as part of the original schema but excluded from the observed state. In particular, we avoid balance artifacts associated with canceled fraudulent transactions described by the dataset provider.

In [ ]:
EXPECTED = {'step','type','amount','nameOrig','oldbalanceOrg','newbalanceOrig',
            'nameDest','oldbalanceDest','newbalanceDest','isFraud','isFlaggedFraud'}
NUMERIC = ['step','amount','oldbalanceOrg','newbalanceOrig',
           'oldbalanceDest','newbalanceDest','isFraud','isFlaggedFraud']
sample_rng = np.random.default_rng(SEED)
retained, priorities, raw_rows = None, None, 0
if MAX_ROWS < 1000: raise ValueError('Use at least 1000 sampled rows; small splits may still be insufficient.')
for chunk in pd.read_csv(DATA_PATH, chunksize=200_000):
    if set(chunk.columns) != EXPECTED: raise ValueError('Unexpected PaySim schema.')
    if chunk.empty or chunk.isna().any().any(): raise ValueError('Missing PaySim values.')
    for column in NUMERIC: chunk[column] = pd.to_numeric(chunk[column], errors='raise')
    if not np.isfinite(chunk[NUMERIC].to_numpy(dtype=float)).all(): raise ValueError('Nonfinite data.')
    if not chunk.type.isin(TYPE_ORDER).all(): raise ValueError('Unknown transaction type.')
    if not chunk[['isFraud','isFlaggedFraud']].isin([0,1]).all().all(): raise ValueError('Nonbinary labels.')
    if (chunk.amount < 0).any() or (chunk.step < 1).any() or (chunk.step % 1 != 0).any():
        raise ValueError('Need nonnegative amounts and positive integer steps.')
    chunk.index = np.arange(raw_rows, raw_rows+len(chunk))
    raw_rows += len(chunk)
    keys = sample_rng.random(len(chunk))
    retained = chunk if retained is None else pd.concat([retained, chunk])
    priorities = keys if priorities is None else np.concatenate([priorities, keys])
    if len(retained) > MAX_ROWS:
        keep = np.argpartition(priorities, MAX_ROWS-1)[:MAX_ROWS]
        retained, priorities = retained.iloc[keep], priorities[keep]
frame = retained.sort_index().copy()
parts, contexts, amount_edges, boundaries = prepare_data(frame)
for name, part in parts.items():
    episode_count(len(part), EPISODE_LENGTH, REVIEW_BUDGET)
    assert part.index.is_unique
assert set(parts['train'].index).isdisjoint(parts['test'].index)
with DATA_PATH.open('rb') as handle:
    source_hash = hashlib.file_digest(handle, 'sha256').hexdigest()
manifest = {'dataset':'ealaxi/paysim1', 'source':str(DATA_PATH.resolve()),
            'sha256':source_hash, 'raw_rows':raw_rows, 'sample_rows':len(frame),
            'sampling':'uniform random-priority across full CSV; then stable chronological sort',
            'python':platform.python_version(), 'versions':versions, 'sample_seed':SEED,
            'training_seeds':TRAIN_SEEDS, 'step_boundaries':boundaries,
            'amount_log_bin_edges':amount_edges.tolist(), 'type_order':TYPE_ORDER,
            'episode_length':EPISODE_LENGTH, 'review_budget':REVIEW_BUDGET,
            'reward_points':{f'{action}_{label}':v for (action,label),v in REWARDS.items()},
            'feedback':'simulated immediate complete labels after every training action',
            'gamma':GAMMA, 'epochs':EPOCHS, 'candidate_alphas':ALPHAS}
split_rows=[]
for name, part in parts.items():
    n = len(part)//EPISODE_LENGTH
    split_rows.append({'split':name, 'rows':len(part), 'first_step':int(part.step.min()),
                      'last_step':int(part.step.max()), 'complete_episodes':n,
                      'evaluated_rows':n*EPISODE_LENGTH, 'dropped_tail_rows':len(part)%EPISODE_LENGTH})
split_audit = pd.DataFrame(split_rows)
manifest['splits'] = split_audit.to_dict(orient='records')
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest, indent=2))
split_audit.to_csv(OUTPUT/'split_audit.csv', index=False)
print('Raw rows / sample:', raw_rows, len(frame)); display(split_audit)
print('Observed policy inputs: type code, training-fitted amount bin, remaining slots, episode position.')
print('Training log-amount bin edges:', amount_edges)
display(parts['train'][['step','type','amount']].head())

## 6 · Training and exploration

One epoch replays every complete training episode once. Episode order is shuffled; transaction order within an episode stays chronological. Epsilon falls from 1 toward 0.05. The table persists between epochs; the budget resets for each episode.

We compare alpha 0.1 and 0.3 using the same three seeds and validation episodes. Gamma, epochs, bins, utility and budgets are fixed before evaluation. We choose alpha by **mean undiscounted validation utility across seeds**. The same training data is replayed repeatedly: more updates do not mean more independent examples.

Training reward includes exploration and is not held-out performance. Do not select a seed because it looks best on the final test. The first predeclared seed supplies the detailed example trace, while all three seeds are reported.

In [ ]:
training_labels = parts['train'].isFraud.to_numpy(dtype=int)
validation_labels = parts['validation'].isFraud.to_numpy(dtype=int)
trained, histories, validation_rows = {}, [], []
for alpha in ALPHAS:
    for seed in TRAIN_SEEDS:
        q, history = train_q(contexts['train'], training_labels,
                             episode_length=EPISODE_LENGTH, budget=REVIEW_BUDGET,
                             epochs=EPOCHS, alpha=alpha, gamma=GAMMA, seed=seed)
        trained[(alpha, seed)] = q
        histories.append(history.assign(alpha=alpha, seed=seed))
        metrics, _ = evaluate_policy(contexts['validation'], validation_labels,
            parts['validation'].amount.to_numpy(), 'q', q=q,
            episode_length=EPISODE_LENGTH, budget=REVIEW_BUDGET, seed=seed)
        validation_rows.append({'alpha':alpha, 'seed':seed, **metrics})
        print(f'alpha={alpha}, seed={seed}, states={len(q):,}, validation utility={metrics["utility"]:.0f}')
training_history = pd.concat(histories, ignore_index=True)
validation_results = pd.DataFrame(validation_rows)
selection = validation_results.groupby('alpha').utility.mean().sort_values(ascending=False, kind='stable')
selected_alpha = float(selection.index[0])
manifest['selected_alpha'] = selected_alpha
manifest['selection_rule'] = 'maximum mean validation undiscounted utility; ties favor first alpha'
manifest['representative_seed'] = TRAIN_SEEDS[0]
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest, indent=2))
training_history.to_csv(OUTPUT/'training_history.csv', index=False)
validation_results.to_csv(OUTPUT/'validation_metrics.csv', index=False)
display(validation_results[['alpha','seed','utility','reviews','precision','recall','unseen_state_share']])
print('Frozen alpha:', selected_alpha)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
for (alpha, seed), h in training_history.groupby(['alpha','seed']):
    label = f'alpha={alpha}, seed={seed}'
    axes[0].plot(h.epoch,h.mean_episode_reward,label=label,alpha=.7)
    axes[1].plot(h.epoch,h.mean_episode_reviews,alpha=.7)
    axes[2].plot(h.epoch,h.visited_states,alpha=.7)
axes[0].set_title('Training utility (with exploration)'); axes[0].set_ylabel('Mean utility / episode')
axes[1].set_title('Training review usage'); axes[1].axhline(REVIEW_BUDGET,color='black',ls='--')
axes[2].set_title('Visited Q-table states')
for ax in axes: ax.set_xlabel('Epoch')
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT/'training_diagnostics.png',dpi=140); plt.show()

## 7 · Fair final comparison

Freeze learning and exploration. Each policy sees the same complete test episodes, in the same order, with the same capacity. Every trailing partial episode is excluded for every policy and reported in `split_audit.csv`.

| Policy | Decision rule | Future information? |
|---|---|---|
| Q-learning | Greedy legal action from the frozen table | No |
| Never review | Always PASS | No |
| Random | Uniformly choose 10 review positions per window | No future features or labels; equivalent to an online remaining-capacity rule |
| First available | Review the first 10 transactions | No |
| Amount threshold | Review if amount ≥ the training 95th percentile and slots remain | No; threshold fitted on training only |
| Offline top amount | Review the 10 largest amounts in each whole window | **Yes, future amounts; look-ahead reference only** |

Offline top amount is not an online-deployable competitor and not a theoretical upper bound on utility. A high amount is not a fraud probability. None of these baselines uses test labels to choose actions.

Utility is the undiscounted sum of the declared rewards. Precision = reviewed fraud / reviewed transactions; recall = reviewed fraud / all fraud in evaluated rows. A ratio with a zero denominator is `NaN`, not zero. If there are no frauds, recall cannot establish detection performance.

The model was trained to approximate **discounted** return with gamma 0.99, while the selection and operational comparison here use undiscounted utility. Keep that distinction visible. We do not assume Q-learning will win.

In [ ]:
# First final-test label access: settings above are now frozen.
test_labels = parts['test'].isFraud.to_numpy(dtype=int)
test_amounts = parts['test'].amount.to_numpy()
threshold = float(parts['train'].amount.quantile(1-REVIEW_BUDGET/EPISODE_LENGTH))
manifest['training_amount_threshold'] = threshold
final_rows, representative_traces = [], {}
policies = ['q','never','random','first','threshold','offline_amount']
for seed in TRAIN_SEEDS:
    frozen_q = trained[(selected_alpha, seed)]
    for policy in policies:
        metrics, trace = evaluate_policy(contexts['test'], test_labels, test_amounts,
            policy, q=frozen_q if policy == 'q' else None, threshold=threshold,
            episode_length=EPISODE_LENGTH, budget=REVIEW_BUDGET, seed=seed)
        final_rows.append({'policy':policy,'seed':seed, **metrics})
        if seed == TRAIN_SEEDS[0]: representative_traces[policy] = trace
final_metrics = pd.DataFrame(final_rows)
assert final_metrics.budget_violations.eq(0).all()
assert final_metrics.evaluated_rows.nunique() == 1
assert final_metrics.dropped_rows.nunique() == 1
summary = final_metrics.groupby('policy',sort=False).agg(
    utility_mean=('utility','mean'), utility_std=('utility','std'),
    reviews_mean=('reviews','mean'), precision_mean=('precision','mean'),
    recall_mean=('recall','mean'), frauds_found_mean=('frauds_found','mean'),
    missed_fraud_mean=('missed_fraud','mean'), budget_violations=('budget_violations','max'))
final_metrics.to_csv(OUTPUT/'test_policy_metrics.csv', index=False)
summary.to_csv(OUTPUT/'test_policy_summary.csv')
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
display(summary)
print('Observed variability is across three fixed training/random-policy seeds, not a confidence interval.')
print('Amount threshold fitted on training:', threshold)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
colors=['#227c9d' if name!='offline_amount' else '#999999' for name in summary.index]
axes[0].bar(summary.index,summary.utility_mean,yerr=summary.utility_std.fillna(0),color=colors,capsize=4)
axes[0].set_ylabel('Utility points'); axes[0].set_title('Final-test utility: mean ± seed SD')
axes[1].bar(summary.index,summary.recall_mean,color=colors)
axes[1].set_ylabel('Fraud recall'); axes[1].set_title('Recall on identical evaluated rows')
for ax in axes: ax.tick_params(axis='x',rotation=35)
plt.tight_layout(); plt.savefig(OUTPUT/'policy_comparison.png',dpi=140); plt.show()

## 8 · Inspect capacity, coverage and individual choices

An observed decision reflects the table, available capacity and fallback behavior. It is not a SHAP explanation, causal finding, fraud probability or proof of criminal intent.

All Q-values initially equal zero. PASS wins greedy ties; unseen states also PASS. During exploratory training, REVIEW may still be sampled. At test time, unseen states do not learn from labels. Sparse state coverage is a practical limitation of tabular models: exact position and remaining capacity expand the table, even with only four amount bins.

We inspect seed 42 because it was chosen before evaluation, not because it performed best. The exported trace includes labels **for post-hoc evaluation only**. Do not feed this trace back into the test policy.

In [ ]:
representative_q = trained[(selected_alpha, TRAIN_SEEDS[0])]
trace = representative_traces['q'].copy()
source_ids = parts['test'].index.to_numpy()
trace['source_row_id'] = source_ids[trace.row_position]
trace['type'] = [TYPE_ORDER[int(t)] for t in trace.type_code]
trace['amount'] = test_amounts[trace.row_position]
q_values=[]
for row in trace.itertuples():
    state = (int(row.type_code),int(row.amount_bin),int(row.budget_before),int(row.position))
    q_values.append(representative_q.get(state,np.zeros(2)))
trace[['q_pass','q_review']] = np.asarray(q_values)
trace['action_name'] = np.where(trace.action.eq(REVIEW),'REVIEW','PASS')
trace.to_csv(OUTPUT/'test_decision_trace.csv',index=False)
trace.loc[trace.action.eq(REVIEW)].to_csv(OUTPUT/'review_queue.csv',index=False)
coverage = final_metrics.loc[final_metrics.policy.eq('q'),['seed','unseen_state_share','reviews','frauds_found']]
display(coverage)
display(trace[['episode','position','type','amount','budget_before','action_name',
               'budget_after','seen_state','q_pass','q_review']].head(20))
first_episode = trace.loc[trace.episode.eq(0)]
plt.step(first_episode.position, first_episode.budget_after,where='post')
plt.xlabel('Transaction position'); plt.ylabel('Slots remaining after action')
plt.title('Review capacity through the first test episode')
plt.tight_layout(); plt.savefig(OUTPUT/'budget_trace.png',dpi=140); plt.show()

# Store all selected-seed tables without pickle so the experiment can be inspected.
for seed in TRAIN_SEEDS:
    q = trained[(selected_alpha,seed)]
    states = np.asarray(sorted(q),dtype=np.int32)
    values = np.asarray([q[tuple(s)] for s in states])
    np.savez_compressed(OUTPUT/f'q_table_seed_{seed}.npz', states=states, values=values)
print('Q-table state columns: type_code, amount_bin, remaining_slots, position; value columns: PASS, REVIEW.')
print('Written:', sorted(p.name for p in OUTPUT.iterdir()))

## 9 · Interpreting the result honestly

Use the generated comparison table rather than assuming a winner. A policy can outperform never-review utility yet miss most fraud; a policy can use every slot and still perform poorly. Compare precision, recall, reviews, missed fraud, utility, unseen states and seed variation together.

**If Q-learning loses:** inspect the rare reward signal, repeated replay, sparse state coverage, unseen-state PASS fallback and whether amount/type carry enough information. Finite data, an approximate state and fixed learning rates do not justify a convergence claim. Repeatedly changing choices after viewing final-test results turns that test into validation; obtain a fresh holdout for further model development.

**Operational gap:** production requires measured review costs, delayed/selected labels, investigation effectiveness, drift monitoring, a justified review-capacity schedule, a way to assess counterfactual decisions, and human governance. This replay is neither evidence of prevented losses nor a deployment-ready review system.

## 10 · Exercises and investigation notes

Run the baseline lesson first. For experiments, create a separate run directory and preserve the original final-test results. Use training and validation for development; do not repeatedly optimize against this test.

1. **State:** List every component in the state. Why is remaining capacity essential? What history has been omitted?
2. **One update:** With old Q=1, reward=−1, alpha=0.2, gamma=0.9 and legal next maximum=3, compute the new Q. Then repeat for a terminal transition.
3. **Masking:** Why must REVIEW be excluded from both action selection and the next-state maximum at zero budget?
4. **Exploration:** Explain the difference between changing epsilon and changing alpha. Why is epsilon zero on test?
5. **Feedback:** Which part of the simulator can see training labels? Why would passing a transaction in reality not necessarily reveal a trustworthy label?
6. **Capacity:** Propose a validation-only experiment with five slots instead of ten. Which policies and denominators must change together?
7. **Rewards:** Increase the legitimate-review cost on training/validation. Explain how the preferred policy might change without claiming a test improvement.
8. **Discount:** Compare gamma=0 and gamma=0.99 on validation. What does gamma=0 ignore? Why is undiscounted evaluation not the same objective as discounted training?
9. **Coverage:** Calculate the observed-state fraction. Would more bins reduce or increase sparsity? Why might an unseen-state PASS fallback be dangerous operationally?
10. **Baselines:** Why is offline top amount allowed only as a look-ahead reference? Does it provide a utility upper bound?
11. **Leakage:** Explain why fitting bins on all rows or choosing the best final-test seed invalidates the comparison.
12. **Metrics:** Give an example where precision rises while recall falls. What should precision be when a policy reviews nothing?
13. **Fairness of comparison:** Why do all policies exclude the exact same incomplete tail windows? What could go wrong if fraud is concentrated there?
14. **Beyond this lab:** What data would be needed to model actual delays, reviewer accuracy and intervention outcomes?

### Three analyst case notes

Choose a reviewed transaction, a passed transaction with slots available, and a passed transaction after budget exhaustion. For each, record observed state, legal actions, Q-values/fallback, decision, capacity effect, and one verification step. Reveal the label only after explaining the decision. A missed fraud is a limitation to report, not a result to conceal.

### Assessment rubric (20 points)

| Area | Points | Evidence |
|---|---:|---|
| RL concepts and worked update | 4 | Correct state/action/reward explanation and arithmetic |
| Environment and capacity | 4 | Trace and terminal/masking checks |
| Experimental design | 4 | Training-only transforms, chronological splits, frozen comparison |
| Interpretation | 4 | Utility, precision, recall, sparsity and baseline discussion |
| Responsible conclusions | 4 | Feedback assumptions, three case notes and deployment limitations |

In [ ]:
# Student response cell: complete your calculations, reasoning and three case notes here.

## References and next steps

- Watkins & Dayan (1992), [Q-learning](https://www.gatsby.ucl.ac.uk/~dayan/papers/cjch.pdf): original algorithm and convergence conditions. The finite experiment here does not establish those conditions.
- Sutton & Barto, [Reinforcement Learning: An Introduction](https://incompleteideas.net/book/the-book-2nd.html), second edition, chapter 6: temporal-difference learning and off-policy control.
- [PaySim dataset and provider notes](https://www.kaggle.com/datasets/ealaxi/paysim1): simulated mobile-money data, schema and balance-field caveats.
- [Session 7](../session-07/README.md): anomaly ranking and SHAP explain different outputs; a Q-table here estimates action value under simulated utility.

Generated artifacts are local. Use `README.md` for launch instructions and `VALIDATION.md` for the exact verification scope and recorded results.